### Data analysis, cleaning and preprocessing

#### Load and analyse the data

In [395]:
import pandas as pd
df = pd.read_csv("../data/raw/characters.csv")
df.head()

,name,full_name,description,categories
0,"Abbot, Hannah","Abbot, Hannah",A Hufflepuff girl in Harry’s year and a prefec...,"['Hufflepuff', 'Dumbledore’s Army']"
1,"Abercrombie, Euan","Abercrombie, Euan",A Gryffindor student who first came to Hogwart...,['Gryffindor']
2,"Ackerley, Steward","Ackerley, Steward",A Ravenclaw student who was first introduced i...,['Ravenclaw']
3,"Aesalon, Falcon","Aesalon, Falcon","From ancient Greece, he was the first ever rec...",['Chocolate Frog Card']
4,Agnes,Agnes,A patient in St. Mungo’s who’s head is covered...,[]


In [396]:
# Remove null values
df.isna().sum()

name           39
full_name      39
description     1
categories      0
dtype: int64

In [397]:
len(df)

697

In [398]:
df[df["name"] .isna()]

,name,full_name,description,categories
363,NaN,NaN,Mangorian- A centaur who lives in the Forbidde...,['Chocolate Frog Card']
383,NaN,NaN,Merlin- Found on a wizard card and the most fa...,['Chocolate Frog Card']
385,NaN,NaN,Merwyn the Malicious- Known for creating jinxe...,['Chocolate Frog Card']
390,NaN,NaN,Moaning Myrtle- A ghost of a Hogwarts student ...,['Ravenclaw']
392,NaN,NaN,Montague- Chaser and Captain for the Slytherin...,"['quidditch', 'Slytherin']"
396,NaN,NaN,"Moony- Remus Lupin’s (see Lupin, Remus J.) nic...","['Order of the Phoenix', 'Hogwarts Staff', 'Gr..."
398,NaN,NaN,Mopsus- Defeated the Seer Calchas.,['Chocolate Frog Card']
399,NaN,NaN,Moran- Chaser for the Irish National Quidditch...,['quidditch']
401,NaN,NaN,"Morgana- See Le Fe, Morgana",['Chocolate Frog Card']
402,NaN,NaN,Morholt- Giant brother of the King of Ireland.,['Chocolate Frog Card']


In [399]:
df[df["full_name"] .isna()]

,name,full_name,description,categories
363,NaN,NaN,Mangorian- A centaur who lives in the Forbidde...,['Chocolate Frog Card']
383,NaN,NaN,Merlin- Found on a wizard card and the most fa...,['Chocolate Frog Card']
385,NaN,NaN,Merwyn the Malicious- Known for creating jinxe...,['Chocolate Frog Card']
390,NaN,NaN,Moaning Myrtle- A ghost of a Hogwarts student ...,['Ravenclaw']
392,NaN,NaN,Montague- Chaser and Captain for the Slytherin...,"['quidditch', 'Slytherin']"
396,NaN,NaN,"Moony- Remus Lupin’s (see Lupin, Remus J.) nic...","['Order of the Phoenix', 'Hogwarts Staff', 'Gr..."
398,NaN,NaN,Mopsus- Defeated the Seer Calchas.,['Chocolate Frog Card']
399,NaN,NaN,Moran- Chaser for the Irish National Quidditch...,['quidditch']
401,NaN,NaN,"Morgana- See Le Fe, Morgana",['Chocolate Frog Card']
402,NaN,NaN,Morholt- Giant brother of the King of Ireland.,['Chocolate Frog Card']


In [400]:
df[df["description"].isna()]

,name,full_name,description,categories
687,NaN,NaN,NaN,[]


We can see that a lot of the null values in the name column are due to the name being attached to the description. We will remove only the rows where all the fields are null and then fix the ones where the name can be found in the description.

In [401]:
df = df.dropna(subset=["name", "full_name", "description"], how="all")

In [402]:
len(df)

696

In [403]:
# Now let's add the names from the description if name or full_name is missing

def extract_name_from_description(row):
    if pd.isna(row["name"]) or pd.isna(row["full_name"]):
        desc = row["description"]
        if pd.notna(desc):
            head, sep, tail = desc.partition("- ")
            if sep:  # we actually found the separator
                head = head.strip()
                if pd.isna(row["name"]):
                    row["name"] = head
                if pd.isna(row["full_name"]):
                    row["full_name"] = head
                row["description"] = tail.strip()
    return row


In [404]:
df.iloc[385]

name                                                         NaN
full_name                                                    NaN
description    Merwyn the Malicious- Known for creating jinxe...
categories                               ['Chocolate Frog Card']
Name: 385, dtype: object

In [405]:
extract_name_from_description(df.iloc[385])


name                            Merwyn the Malicious
full_name                       Merwyn the Malicious
description    Known for creating jinxes and curses.
categories                   ['Chocolate Frog Card']
Name: 385, dtype: object

In [406]:
df = df.apply(extract_name_from_description, axis=1)
df.iloc[385]

name                            Merwyn the Malicious
full_name                       Merwyn the Malicious
description    Known for creating jinxes and curses.
categories                   ['Chocolate Frog Card']
Name: 385, dtype: object

In [407]:
# Check that there are no more missing names
df[df["name"] .isna()]

,name,full_name,description,categories
647,NaN,NaN,OTHER WEASLEYS,[]


In [408]:
# Let's manually remove that line
df = df.drop(index=647)

In [409]:
# Check that there are no more missing full names
df[df["full_name"].isna()]

,name,full_name,description,categories


In [410]:
# Check that there are no more null descriptions
df[df["description"].isna()]

,name,full_name,description,categories


Now that we removed the null values, let's examine the names and make sure they're formatted correctly.

In [411]:
# Let's check for non-alphanumeric characters in names
non_alphanum_filter = df["name"].str.contains(r"[^a-zA-Z0-9\s]", regex=True)
df[non_alphanum_filter]

,name,full_name,description,categories
0,"Abbot, Hannah","Abbot, Hannah",A Hufflepuff girl in Harry’s year and a prefec...,"['Hufflepuff', 'Dumbledore’s Army']"
1,"Abercrombie, Euan","Abercrombie, Euan",A Gryffindor student who first came to Hogwart...,['Gryffindor']
2,"Ackerley, Steward","Ackerley, Steward",A Ravenclaw student who was first introduced i...,['Ravenclaw']
3,"Aesalon, Falcon","Aesalon, Falcon","From ancient Greece, he was the first ever rec...",['Chocolate Frog Card']
5,"Agrippa, Cornelius","Agrippa, Cornelius",1535 and was on a wizard card saying celebrate...,['Chocolate Frog Card']
...,...,...,...,...
684,"Worple, Eldred","Worple, Eldred",An old friend of Slughorn who is the author of...,[]
688,? Yaxley,? Yaxley,A death eater who was never sent to Azkaban.,['Death Eater']
689,Lysandra- Wife of Arcturus Black who lived 188...,Lysandra- Wife of Arcturus Black who lived 188...,"1959. Had two daughters, Callidora and Charis;...",[]
693,"Zabini, ?","Zabini, ?",Was famous for being beautiful and marrying 7 ...,[]


In [412]:
non_alphanum_filter.sum()

361

In [413]:
# grab all non-alphanum chars per row (NaNs -> empty string)
chars = (
    df["full_name"]
    .str.findall(r"[^a-zA-Z0-9\s]")
    .explode()
    .dropna()
    .unique()
)

print(chars)

[',' '(' '’' ')' '.' '?' '“' '”' '-']


In [414]:
df[non_alphanum_filter]["full_name"].unique(), non_alphanum_filter.sum()

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. ?s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, ? (Grandmother of Susan B

There are several problems:
1) Description included in the name (includes "-")
2) Question marks and extra spaces
3) The "Black" and "Weasly" surnames in parenthesis instead of preceding the name with a comma like the rest of the df
4) Nicknames in parenthesis
5) Quote marks in the names with descriptions

Let's examine each case individually and clean the data for each issue independently.

In [415]:
# Check that the rows where the name has a description doesn't have an empty description
df[df["full_name"].str.contains("-")]

,name,full_name,description,categories
135,"Delaney- Podmore, Patrick","Delaney- Podmore, Patrick",(Sir Patrick Delaney-Podmore) Leader of the He...,[]
181,Ethelred the Ever-Ready,Ethelred the Ever-Ready,Wizard who was famous for being offended easil...,['Chocolate Frog Card']
199,"Finch-Fletchley, Justin","Finch-Fletchley, Justin",A muggle born Hufflepuff boy in Harry’s year. ...,"['Hufflepuff', 'Dumbledore’s Army']"
249,"Grubbly-Plank, Professor","Grubbly-Plank, Professor",A witch who temporarily substitutes for Hagrid...,['Hogwarts Staff']
357,Draco - The son of Lucius and Narcissa Malfoy ...,Draco - The son of Lucius and Narcissa Malfoy ...,the Blacks and the Malfoys. He does not think ...,"['quidditch', 'Slytherin', 'Related to the Bla..."
388,Sir Nicholas de Mimsy-Porpington,Sir Nicholas de Mimsy-Porpington,The full name of the Gryffindor House Ghost (m...,['Gryffindor']
394,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,Eye Moody. A retired auror from the ministry o...,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi..."
419,Phineas- The great Nigellus,Phineas- The great Nigellus,great grandfather of Sirius Black who was (as ...,"['Hogwarts Staff', 'Slytherin', 'Related to th..."
423,Theodore- A “weedy Nott,Theodore- A “weedy Nott,looking” student who hangs out with Draco Malf...,['Slytherin']
450,Jimmy- A boy described as short but broad Peakes,Jimmy- A boy described as short but broad Peakes,chested and in his third year. He made the Gry...,"['quidditch', 'Gryffindor']"


In [416]:
# Some of the instances are nicknames, not descriptions, so let's filter those out and just work on the descriptions in this step
nickname_indexes = [135, 181, 199, 249, 388, 394, 459]

In [417]:
# 1 Remove descriptions from names
def remove_description_from_name(row):
    name = row["full_name"]
    if pd.notna(name) and row.name not in nickname_indexes:
        head, sep, tail = name.partition("- ")
        if sep:  # we actually found the separator
            row["full_name"] = head.strip()
        row["description"] = tail.strip()
    return row

df = df.apply(remove_description_from_name, axis=1)
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. ?s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, ? (Grandmother of Susan B

In [418]:
# 2 Remove question marks and extra spaces
def clean_question_marks_and_spaces(row):
    name = row["full_name"]
    if pd.notna(name):
        # Remove question marks
        name = name.replace("?", "")
        # Remove extra spaces
        name = " ".join(name.split())
        row["full_name"] = name
    return row

df = df.apply(clean_question_marks_and_spaces, axis=1)  
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', '(Black) Andromeda', '(Black) Araminta Meliflua',
        '(Black) Bellatrix Lestrange', '(Black) Aunt Elladora',
        '(Black) Narcissa', 'Black, Regulus', 'Black, Sirius',
        '(Black) Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, (Grandmother of Susan Bone

In [419]:
df[df["full_name"].str.contains("-")]

,name,full_name,description,categories
135,"Delaney- Podmore, Patrick","Delaney- Podmore, Patrick",(Sir Patrick Delaney-Podmore) Leader of the He...,[]
181,Ethelred the Ever-Ready,Ethelred the Ever-Ready,Wizard who was famous for being offended easil...,['Chocolate Frog Card']
199,"Finch-Fletchley, Justin","Finch-Fletchley, Justin",A muggle born Hufflepuff boy in Harry’s year. ...,"['Hufflepuff', 'Dumbledore’s Army']"
249,"Grubbly-Plank, Professor","Grubbly-Plank, Professor",A witch who temporarily substitutes for Hagrid...,['Hogwarts Staff']
388,Sir Nicholas de Mimsy-Porpington,Sir Nicholas de Mimsy-Porpington,The full name of the Gryffindor House Ghost (m...,['Gryffindor']
394,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,Eye Moody. A retired auror from the ministry o...,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi..."
459,"Perks, Sally-Ann","Perks, Sally-Ann",A student in Harry’s year who was sorted befor...,[]


In [420]:
# 3 Extract surnames from parenthesis and place them before the name with a comma
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
18,"Bagman, (Ludo and Otto’s father)","Bagman, (Ludo and Otto’s father)",,[]
43,(Black) Andromeda,(Black) Andromeda,,['Related to the Blacks']
44,(Black) Araminta Meliflua,(Black) Araminta Meliflua,,['Related to the Blacks']
45,(Black) Bellatrix Lestrange,(Black) Bellatrix Lestrange,,"['Slytherin', 'Death Eater', 'Related to the B..."
46,(Black) Aunt Elladora,(Black) Aunt Elladora,,"['Slytherin', 'Related to the Blacks']"
47,(Black) Narcissa,(Black) Narcissa,,"['Slytherin', 'Death Eater', 'Related to the B..."
50,(Black) Uncle Alfred,(Black) Uncle Alfred,,['Related to the Blacks']
64,"Bones, ? (Grandmother of Susan Bones)","Bones, (Grandmother of Susan Bones)",,[]
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
212,Flourish and Blotts (Manager),Flourish and Blotts (Manager),,[]


In [421]:
# There are more cases of parenthesis, so let's make a replacement dict for the general cases and then examine the exceptions individually
import re
replacement_dict = {
    r"\(Black\)": "Black, ",
    r"\(Weasly\)": "Weasly, ",
    r"\(Granger\)": "Granger, ",
    r"\(Weasly\)\,": "Weasly, ",
    r"\(Delacour\)": "Delacour, ",
    r"\(\)": "",
}

def replace_parenthesis_surnames(row):
    name = row["full_name"]
    if pd.notna(name):
        for pattern, replacement in replacement_dict.items():
            name = re.sub(pattern, replacement, name)
        row["full_name"] = name
    return row

df = df.apply(replace_parenthesis_surnames, axis=1) 
df[non_alphanum_filter]["full_name"].unique(), df[non_alphanum_filter].shape[0]

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman, (Ludo and Otto’s father)',
        'Bagnold, Millicent', 'Bagshot, Bathilda', 'Barbary, Heathcote',
        'Barkwith, Musidora', 'Bashir, Ali', 'Bayliss, Hetty',
        'Beamish, Oswald', 'Belby, Flavius', 'Belby, Marcus',
        'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
        'Bishop, Dennis', 'Black,  Andromeda', 'Black,  Araminta Meliflua',
        'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
        'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
        'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar',
        'Bones, (Grandmother of Susan Bone

In [422]:
df.iloc[43]

name                   (Black) Andromeda
full_name              Black,  Andromeda
description                             
categories     ['Related to the Blacks']
Name: 43, dtype: object

In [423]:
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
18,"Bagman, (Ludo and Otto’s father)","Bagman, (Ludo and Otto’s father)",,[]
64,"Bones, ? (Grandmother of Susan Bones)","Bones, (Grandmother of Susan Bones)",,[]
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
212,Flourish and Blotts (Manager),Flourish and Blotts (Manager),,[]
226,(Guant) Tom Marvolo,(Guant) Tom Marvolo,,"['Slytherin', 'Death Eater']"
227,Ghoul (of the Weasleys),Ghoul (of the Weasleys),,[]
228,Giant Squid (of Hogwarts),Giant Squid (of Hogwarts),,[]
242,Grey Lady (Helena Ravenclaw),Grey Lady (Helena Ravenclaw),,['Ravenclaw']
341,Nymphadora (Dora) Tonks Lupin,Nymphadora (Dora) Tonks Lupin,,[]
412,Muriel (Aunt),Muriel (Aunt),,[]


In [424]:
# Let's handle the descriptions that are still in parenthesis
def handle_descriptions_in_parenthesis(row):
    name = row["full_name"]
    if pd.notna(name):
        head, sep, tail = name.partition(" (")
        if sep:  # we actually found the separator
            description = tail.rstrip(")")
            row["full_name"] = head.strip()
            if pd.isna(row["description"]) or row["description"] == "":
                row["description"] = description.strip()
    return row

desc_indexes = [18, 64, 212]
df.iloc[desc_indexes] = df.iloc[desc_indexes].apply(handle_descriptions_in_parenthesis, axis=1)



In [425]:
df[df["full_name"].str.contains(r"\(.*\)")]

,name,full_name,description,categories
167,"Dursley, Marjorie (Marge)","Dursley, Marjorie (Marge)",,[]
226,(Guant) Tom Marvolo,(Guant) Tom Marvolo,,"['Slytherin', 'Death Eater']"
227,Ghoul (of the Weasleys),Ghoul (of the Weasleys),,[]
228,Giant Squid (of Hogwarts),Giant Squid (of Hogwarts),,[]
242,Grey Lady (Helena Ravenclaw),Grey Lady (Helena Ravenclaw),,['Ravenclaw']
341,Nymphadora (Dora) Tonks Lupin,Nymphadora (Dora) Tonks Lupin,,[]
412,Muriel (Aunt),Muriel (Aunt),,[]
514,Helena (The Grey Lady) Ravenclaw,Helena (The Grey Lady) Ravenclaw,,['Ravenclaw']
535,Newton (Newt) Artemis Fido Scamander,Newton (Newt) Artemis Fido Scamander,,"['Chocolate Frog Card', 'Ministry of Magic']"
643,Ginevra (Ginny) Weasley,Ginevra (Ginny) Weasley,,"['Quidditch Player', 'gryff', 'Slug Club', 'Re..."


In [426]:
# Separate the nicknames from the full names with commas
def extract_nickname(row):
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or not full_name:
        return row

    matches = re.findall(r"\(([^)]+)\)", full_name)
    if not matches:
        return row

    nickname = ", ".join(matches)
    cleaned = re.sub(r"\s*\(([^)]+)\)", "", full_name)
    cleaned = " ".join(cleaned.split())

    row["full_name"] = f"{cleaned}, {nickname}"
    row["nickname"] = nickname  # optional new column

    return row

extract_nickname(df.iloc[643])

name                                     Ginevra (Ginny) Weasley
full_name                                 Ginevra Weasley, Ginny
description                                                     
categories     ['Quidditch Player', 'gryff', 'Slug Club', 'Re...
nickname                                                   Ginny
Name: 643, dtype: object

In [427]:
df = df.apply(extract_nickname, axis=1)

In [428]:
df[df["full_name"].str.contains(r"\(.*\)")]

,categories,description,full_name,name,nickname


In [429]:
# Remove quote marks
df[df["full_name"].str.contains(r'“|”|’')] 

,categories,description,full_name,name,nickname
80,[],,"Brook Stanton, Rupert “Axebanger”","Brook Stanton, Rupert “Axebanger”",NaN
307,[],,Kevin’s mother,Kevin’s mother,NaN
343,[],,Ted “Teddy” Remus Lupid,Ted “Teddy” Remus Lupid,NaN
431,['quidditch'],,Darren O’Hare,Darren O’Hare,NaN
634,[],Street peddler who was accused by the Ministry...,“Honest Willy”,“Honest Willy”- Street peddler who was accused...,NaN


In [430]:
# The quotes that remain are nicknames, so let's extract them
def extract_nickname_from_quotes(row):
    full_name = row.get("full_name")
    if not isinstance(full_name, str) or not full_name:
        return row

    matches = re.findall(r"\“([^”]+)\”", full_name)
    if not matches:
        return row

    nickname = ", ".join(matches)
    cleaned = re.sub(r"\s*\“[^”]+\”", "", full_name)
    cleaned = " ".join(cleaned.split())

    row["full_name"] = f"{cleaned}, {nickname}"
    row["nickname"] = nickname  # optional new column

    return row

In [431]:
df = df.apply(extract_nickname_from_quotes, axis=1)
df[df["full_name"].str.contains(r'“|”|’')] 

,categories,description,full_name,name,nickname
307,[],,Kevin’s mother,Kevin’s mother,NaN
431,['quidditch'],,Darren O’Hare,Darren O’Hare,NaN


In [432]:
df.iloc[343]

categories                          []
description                           
full_name       Ted Remus Lupid, Teddy
name           Ted “Teddy” Remus Lupid
nickname                         Teddy
Name: 343, dtype: object

In [433]:
# Let's check how the full names with special characters look now
df[non_alphanum_filter]["full_name"].unique(), non_alphanum_filter.sum()

(array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
        'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
        'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
        'Bagman, Otto', 'Bagman,', 'Bagnold, Millicent',
        'Bagshot, Bathilda', 'Barbary, Heathcote', 'Barkwith, Musidora',
        'Bashir, Ali', 'Bayliss, Hetty', 'Beamish, Oswald',
        'Belby, Flavius', 'Belby, Marcus', 'Belcher, Humphrey',
        'Bell, Katie', 'Benson, Amy', 'Bishop, Dennis',
        'Black,  Andromeda', 'Black,  Araminta Meliflua',
        'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
        'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
        'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
        'Blott. s', 'Bloxam, Beatrix', 'Bludd, Blodwyn',
        'Boardman, Stubby', 'Bode, Broderick', 'Bonnacord, Pierre',
        'Bones, Amelia', 'Bones, Edgar', 'Bones,', 'Bones, Susan',
        'Bonham, Mungo', 'Boot, Terry', '

In [434]:
# grab all non-alphanum chars per row (NaNs -> empty string)
chars = (
    df["full_name"]
    .str.findall(r"[^a-zA-Z0-9\s]")
    .explode()
    .dropna()
    .unique()
)

print(chars)

[',' '.' '-' '’']


Now let's check how the special characters we got look.

In [435]:
df[df["full_name"].str.contains(r"\,")]["full_name"].unique()

array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
       'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
       'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
       'Bagman, Otto', 'Bagman,', 'Bagnold, Millicent',
       'Bagshot, Bathilda', 'Barbary, Heathcote', 'Barkwith, Musidora',
       'Bashir, Ali', 'Bayliss, Hetty', 'Beamish, Oswald',
       'Belby, Flavius', 'Belby, Marcus', 'Belcher, Humphrey',
       'Bell, Katie', 'Benson, Amy', 'Bishop, Dennis',
       'Black,  Andromeda', 'Black,  Araminta Meliflua',
       'Black,  Bellatrix Lestrange', 'Black,  Aunt Elladora',
       'Black,  Narcissa', 'Black, Regulus', 'Black, Sirius',
       'Black,  Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
       'Bloxam, Beatrix', 'Bludd, Blodwyn', 'Boardman, Stubby',
       'Bode, Broderick', 'Bonnacord, Pierre', 'Bones, Amelia',
       'Bones, Edgar', 'Bones,', 'Bones, Susan', 'Bonham, Mungo',
       'Boot, Terry', 'Boothby, Gladys', 'Borage, L

In [436]:
# Let's remove the commans that are not separating surnames or nicknames
df["full_name"] = (
    df["full_name"]
    .str.replace(r"^\s*,\s*", "", regex=True)   # leading commas/spaces
    .str.replace(r"\s*,\s*$", "", regex=True)   # trailing commas/spaces
    .str.replace(r"\s*,\s*", ", ", regex=True)  # collapse inner whitespace
    .str.replace(r"\s+", " ", regex=True)       # collapse spaces in general
    .str.strip()
)

df[df["full_name"].str.contains(r"\,")]["full_name"].unique()


array(['Abbot, Hannah', 'Abercrombie, Euan', 'Ackerley, Steward',
       'Aesalon, Falcon', 'Agrippa, Cornelius', 'Alderton, Archibald',
       'Aubrey, Bertram', 'Baddock, Malcolm', 'Bagman, Ludo',
       'Bagman, Otto', 'Bagnold, Millicent', 'Bagshot, Bathilda',
       'Barbary, Heathcote', 'Barkwith, Musidora', 'Bashir, Ali',
       'Bayliss, Hetty', 'Beamish, Oswald', 'Belby, Flavius',
       'Belby, Marcus', 'Belcher, Humphrey', 'Bell, Katie', 'Benson, Amy',
       'Bishop, Dennis', 'Black, Andromeda', 'Black, Araminta Meliflua',
       'Black, Bellatrix Lestrange', 'Black, Aunt Elladora',
       'Black, Narcissa', 'Black, Regulus', 'Black, Sirius',
       'Black, Uncle Alfred', 'Blane, Balfour', 'Bletchley, Miles',
       'Bloxam, Beatrix', 'Bludd, Blodwyn', 'Boardman, Stubby',
       'Bode, Broderick', 'Bonnacord, Pierre', 'Bones, Amelia',
       'Bones, Edgar', 'Bones, Susan', 'Bonham, Mungo', 'Boot, Terry',
       'Boothby, Gladys', 'Borage, Libatius', 'Bott, Bertie',
       '

In [437]:
df[df["full_name"].str.contains("Zabini")]

,categories,description,full_name,name,nickname
692,"['slyth', 'Slug Club']",,Blaise Zabini,Blaise Zabini,NaN
693,[],,Zabini,"Zabini, ?",NaN


In [438]:
# Now let's check the period characters
df[df["full_name"].str.contains(r"\.")]["full_name"].unique()

array(['Blott. s', 'Bunky, K.', 'Capper, S.', 'Crouch, Barty Sr.',
       'Crouch, Barty Jr.', 'Dorny, J.', 'Fawcett, S.', 'King, R.J.H.',
       'Remus J. Lupin', 'M. G. McGonagall', 'Mr. Paws', 'Norris, Mrs.',
       'Paws, Mr.', 'Prod, D. J.'], dtype=object)

They are all full names or abbreviations of the names so they look fine, let's move on to the dashes.

In [439]:
# Let's check how the dashes look
df[df["full_name"].str.contains(r"\-")]["full_name"].unique()

array(['Delaney- Podmore, Patrick', 'Ethelred the Ever-Ready',
       'Finch-Fletchley, Justin', 'Grubbly-Plank, Professor',
       'Sir Nicholas de Mimsy-Porpington', 'Alastor- Nicknamed Mad Moody',
       'Perks, Sally-Ann'], dtype=object)

Most of them are ok let's manually change the mad moody nickname 

In [440]:
df[df["full_name"].str.contains("Alastor")]

,categories,description,full_name,name,nickname
394,"['Order of the Phoenix', 'Hogwarts Staff', 'Mi...",Eye Moody. A retired auror from the ministry o...,Alastor- Nicknamed Mad Moody,Alastor- Nicknamed Mad Moody,NaN


In [441]:
df.at[394, "full_name"] = df.at[394, "full_name"].replace("- Nicknamed ", ", ")
df.at[394, "nickname"] = "Mad Moody"
df.iloc[394]

categories     ['Order of the Phoenix', 'Hogwarts Staff', 'Mi...
description    Eye Moody. A retired auror from the ministry o...
full_name                                     Alastor, Mad Moody
name                                Alastor- Nicknamed Mad Moody
nickname                                               Mad Moody
Name: 394, dtype: object

In [442]:
# Remove the commas
df["full_name"] = df["full_name"].str.replace(r"\s*,\s*", " ", regex=True).str.strip()
df["full_name"].unique()

array(['Abbot Hannah', 'Abercrombie Euan', 'Ackerley Steward',
       'Aesalon Falcon', 'Agnes', 'Agrippa Cornelius',
       'Alderton Archibald', 'Alguf the Awful', 'Alphard', 'Archie',
       'Aragog', 'Arnold', 'Aubrey Bertram', 'Avery', 'Babayaga',
       'Baddock Malcolm', 'Bagman Ludo', 'Bagman Otto', 'Bagman',
       'Bagnold Millicent', 'Bagshot Bathilda', 'Bandon Banshee', 'Bane',
       'Barbary Heathcote', 'Barker', 'Barkwith Musidora',
       'Barnabas the Barmy', 'Barny', 'Barrufio', 'Bashir Ali', 'Basil',
       'Bayliss Hetty', 'Beamish Oswald', 'Beedle the Bard',
       'Belby Flavius', 'Belby Marcus', 'Belcher Humphrey', 'Bell Katie',
       'Benson Amy', 'Billy', 'Binky', 'Binns', 'Bishop Dennis',
       'Black Andromeda', 'Black Araminta Meliflua',
       'Black Bellatrix Lestrange', 'Black Aunt Elladora',
       'Black Narcissa', 'Black Regulus', 'Black Sirius',
       'Black Uncle Alfred', 'Blane Balfour', 'Bletchley Miles',
       'Bloody Barron', 'Blott. s', 'Blo

In [443]:
# Now let's split it and check the ones that have more than 2 parts to analyse further
name_parts = df["full_name"].str.split()
name_parts_counts = name_parts.apply(len)
name_parts_counts.value_counts()    

full_name
2    428
1    192
3     58
4     12
5      5
Name: count, dtype: int64

In [444]:
len(name_parts[name_parts_counts > 2])

75

In [445]:
long_names = name_parts[name_parts_counts > 2].reset_index()
long_names

,index,full_name
0,7,"[Alguf, the, Awful]"
1,26,"[Barnabas, the, Barmy]"
2,33,"[Beedle, the, Bard]"
3,44,"[Black, Araminta, Meliflua]"
4,45,"[Black, Bellatrix, Lestrange]"
...,...,...
70,651,"[Weasley, Hermione, Granger]"
71,655,"[Second, Cousin, Weasley]"
72,660,"[Wendelin, the, Weird]"
73,671,"[Wilfred, the, Wistful]"


In [446]:
# Let's inspect the names that don't contain "the"
mask = long_names["full_name"].apply(lambda parts: "the" in parts if isinstance(parts, list) else
False)

long_names[~mask]  # rows without "the"

,index,full_name
3,44,"[Black, Araminta, Meliflua]"
4,45,"[Black, Bellatrix, Lestrange]"
5,46,"[Black, Aunt, Elladora]"
6,50,"[Black, Uncle, Alfred]"
7,76,"[Brankovitch, III, Maximus]"
8,78,"[Broadmoor, Kevin, and, Karl]"
9,80,"[Brook, Stanton, Rupert, Axebanger]"
10,86,"[Bungs, Rosalind, Antigone]"
11,120,"[Crouch, Barty, Sr.]"
12,121,"[Crouch, Barty, Jr.]"


Some of the names are long but they all look fine.

#### Fix the "name" column

In [447]:
df["full_name"].unique()

array(['Abbot Hannah', 'Abercrombie Euan', 'Ackerley Steward',
       'Aesalon Falcon', 'Agnes', 'Agrippa Cornelius',
       'Alderton Archibald', 'Alguf the Awful', 'Alphard', 'Archie',
       'Aragog', 'Arnold', 'Aubrey Bertram', 'Avery', 'Babayaga',
       'Baddock Malcolm', 'Bagman Ludo', 'Bagman Otto', 'Bagman',
       'Bagnold Millicent', 'Bagshot Bathilda', 'Bandon Banshee', 'Bane',
       'Barbary Heathcote', 'Barker', 'Barkwith Musidora',
       'Barnabas the Barmy', 'Barny', 'Barrufio', 'Bashir Ali', 'Basil',
       'Bayliss Hetty', 'Beamish Oswald', 'Beedle the Bard',
       'Belby Flavius', 'Belby Marcus', 'Belcher Humphrey', 'Bell Katie',
       'Benson Amy', 'Billy', 'Binky', 'Binns', 'Bishop Dennis',
       'Black Andromeda', 'Black Araminta Meliflua',
       'Black Bellatrix Lestrange', 'Black Aunt Elladora',
       'Black Narcissa', 'Black Regulus', 'Black Sirius',
       'Black Uncle Alfred', 'Blane Balfour', 'Bletchley Miles',
       'Bloody Barron', 'Blott. s', 'Blo

In [448]:
df["name"] = df["full_name"].str.split(" ")
df["name"]

0           [Abbot, Hannah]
1       [Abercrombie, Euan]
2       [Ackerley, Steward]
3         [Aesalon, Falcon]
4                   [Agnes]
               ...         
692        [Blaise, Zabini]
693                [Zabini]
694    [Ladislaw, Zamojski]
695          [Rose, Zeller]
696                [Zograf]
Name: name, Length: 695, dtype: object

In [449]:
# ok now let's check for names that are repeated - we need a unique identifier for each character
# 1. Explode the lists so each word becomes its own row
df_exploded = df["name"].explode()

# 2. Find words that appear more than once in the entire column
repeated_words = df_exploded.value_counts()
repeated_words = repeated_words[repeated_words > 1].index

# 3. For each row, store the list of repeated words it contains
df["repeated_words"] = df["name"].apply(
    lambda lst: [word for word in lst if word in repeated_words]
)

# 4. Boolean column: does the row contain any repeated words?
df["has_repeated"] = df["repeated_words"].apply(lambda x: len(x) > 0)


In [450]:
df[df["has_repeated"]==True][["has_repeated", "repeated_words"]]

,has_repeated,repeated_words
5,True,[Cornelius]
7,True,"[the, Awful]"
11,True,[Arnold]
15,True,[Malcolm]
16,True,[Bagman]
...,...,...
680,True,[Lord]
682,True,[Augustus]
692,True,[Zabini]
693,True,[Zabini]


In [451]:
repeated_word_counts = df["repeated_words"].value_counts().reset_index()
repeated_word_counts 

,repeated_words,count
0,[],404
1,[the],13
2,[Weasley],12
3,[Longbottom],5
4,[Madam],4
...,...,...
180,[Flourish],1
181,[Marcus],1
182,[Flamel],1
183,"[Flamel, Nicholas]",1


In [452]:
repeated_word_list = repeated_word_counts["repeated_words"].tolist()
print(repeated_word_list)


[[], ['the'], ['Weasley'], ['Longbottom'], ['Madam'], ['Diggory'], ['Dursley'], ['Smethwyk'], ['Delacour'], ['Scamander'], ['Augustus'], ['Bones'], ['Krum'], ['Dumbledore'], ['Lovegood'], ['Bagman'], ['Malfoy'], ['Lestrange'], ['Lily', 'Potter'], ['McGonagall'], ['McCormack'], ['McClivert'], ['Scrimgeour'], ['Elladora'], ['Sir', 'Nicholas', 'de'], ['Emeric', 'the', 'Evil'], ['Dorcas'], ['Finnigan'], ['Urquhart'], ['Gaunt'], ['of', 'the'], ['Morgan'], ['James', 'Potter'], ['Professor'], ['Gudgeon'], ['Hagrid'], ['Hengist', 'of'], ['Patil'], ['J.'], ['Cornelius'], ['Kevin'], ['Ernie'], ['Myrtle'], ['Ollerton'], ['Montague'], ['Hornby'], ['Rosier'], ['Daisy'], ['Black'], ['Elfrinda'], ['Agatha'], ['Carrow'], ['S.'], ['Lord'], ['Willy'], ['Dennis'], ['Crabbe'], ['Gilbert'], ['Basil'], ['Millicent'], ['Malcolm'], ['Zabini'], ['the', 'Awful'], ['Edgar'], ['McLaggen'], ['Harold'], ['Derwent'], ['Doris'], ['Stebbins'], ['Crouch', 'Barty'], ['Andromeda', 'Tonks'], ['Weasley', 'Granger'], ['Tom'

In [453]:
df.head()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
0,"['Hufflepuff', 'Dumbledore’s Army']",,Abbot Hannah,"[Abbot, Hannah]",NaN,[],False
1,['Gryffindor'],,Abercrombie Euan,"[Abercrombie, Euan]",NaN,[],False
2,['Ravenclaw'],,Ackerley Steward,"[Ackerley, Steward]",NaN,[],False
3,['Chocolate Frog Card'],,Aesalon Falcon,"[Aesalon, Falcon]",NaN,[],False
4,[],,Agnes,[Agnes],NaN,[],False


In [454]:
df.tail()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
692,"['slyth', 'Slug Club']",,Blaise Zabini,"[Blaise, Zabini]",NaN,[Zabini],True
693,[],,Zabini,[Zabini],NaN,[Zabini],True
694,['Quidditch Player'],,Ladislaw Zamojski,"[Ladislaw, Zamojski]",NaN,[],False
695,['Hufflepuff'],,Rose Zeller,"[Rose, Zeller]",NaN,[Rose],True
696,['Quidditch Player'],,Zograf,[Zograf],NaN,[],False


In [460]:
# Remove repeated words from names
def subtract_repeats(row):
    repeated = set(row["repeated_words"] or [])
    tokens = row["name"] if isinstance(row["name"], list) else str(row["name"]).split()
    pruned = [tok for tok in tokens if tok not in repeated]
    row["name"] = pruned if pruned else tokens  # keep original if all tokens repeat
    return row

df = df.apply(subtract_repeats, axis=1)

In [461]:
df.tail()

,categories,description,full_name,name,nickname,repeated_words,has_repeated
692,"['slyth', 'Slug Club']",,Blaise Zabini,[Blaise],NaN,[Zabini],True
693,[],,Zabini,[Zabini],NaN,[Zabini],True
694,['Quidditch Player'],,Ladislaw Zamojski,"[Ladislaw, Zamojski]",NaN,[],False
695,['Hufflepuff'],,Rose Zeller,[Zeller],NaN,[Rose],True
696,['Quidditch Player'],,Zograf,[Zograf],NaN,[],False


In [464]:
# lets check how many unique names we have now
mask = df["name"].apply(lambda tokens: isinstance(tokens, list) and len(tokens) > 1)
df[mask][["full_name", "name", "repeated_words", "has_repeated"]]

,full_name,name,repeated_words,has_repeated
0,Abbot Hannah,"[Abbot, Hannah]",[],False
1,Abercrombie Euan,"[Abercrombie, Euan]",[],False
2,Ackerley Steward,"[Ackerley, Steward]",[],False
3,Aesalon Falcon,"[Aesalon, Falcon]",[],False
6,Alderton Archibald,"[Alderton, Archibald]",[],False
...,...,...,...,...
684,Worple Eldred,"[Worple, Eldred]",[],False
685,Bowman Wright,"[Bowman, Wright]",[],False
686,Josef Wronski,"[Josef, Wronski]",[],False
690,Cyprian Youdle,"[Cyprian, Youdle]",[],False


In [465]:
df.to_csv("../data/processed/characters_cleaned.csv", index=False)

In [ ]:
# TODO:
# ordenar x nombre apellido xq algunos vienen bien y otros no
# eliminar filas duplicadas
# poner el apodo mas significativo 
# quedarse con el nombre si no se repite y si no nombre y apellido


## Final cleanup

In [2]:
import pandas as pd

In [4]:
df = pd.read_csv("../data/processed/characters_final_2.csv")
df.head()

,Unnamed: 0,full_name,categories,name
0,0,Hannah Abbot,"['Hufflepuff', 'Dumbledore’s Army']",Hannah
1,1,Euan Abercrombie,['Gryffindor'],Euan
2,2,Steward Ackerley,['Ravenclaw'],Steward
3,3,Falcon Aesalon,['Chocolate Frog Card'],Falcon
4,4,Agnes,[],Agnes


In [6]:
# Let's separate some rows
df.loc[687] = {"name":"Mrs. Granger", "full_name":"Mrs Granger"}
df.at[686, "name"] = "Mr. Granger"

df.tail()

,Unnamed: 0,full_name,categories,name
683,683.0,Ladislaw Zamojski,['Quidditch Player'],Ladislaw
684,684.0,Rose Zeller,['Hufflepuff'],Rose Zeller
685,685.0,Zograf,['Quidditch Player'],Zograf
686,686.0,Mr and Mrs Granger,NaN,Mr. Granger
687,NaN,Mrs Granger,NaN,Mrs. Granger


In [8]:
df.loc[df["name"] == "Old", "name"] = "Old Mother Hubbard"
df.at[53, "name"] = "Bloody Baron"
df.at[54, "name"] = "Blott"
df.loc[len(df)] = df.iloc[78].copy()
df.at[78, "name"] = "Kevin"
df.at[78, "full_name"] = "Kevin Broadmoor"
df.at[len(df)-1, "name"] = "Karl Broadmoor"
df.at[len(df)-1, "full_name"] = "Karl Broadmoor"
df.at[80, "name"] = "Rupert"
df.at[134, "name"] = "Monsieur Delacour"
df.at[166, "name"] = "Marge"
df.at[355, "name"] = "Narcissa"
df.at[413, "name"] = "Nearly Headless Nick"
df.loc[len(df)] = df.iloc[633].copy()
df.at[633, "full_name"] = "Fred Weasley"
df.at[len(df)-1, "name"] = "George"
df.at[len(df)-1, "full_name"] = "George Weasley"

In [10]:
df.drop(index=[209, 210, 391, 459], inplace=True)

In [12]:
# Let's create a dict with some of the names that refer to the same character
name_corrections = {
    "Bellatrix Lestrange Black": "Bellatrix",
    "Bellatrix Lestrange": "Bellatrix",
    "Sirius Black": "Sirius",
    "Sirius": "Sirius",
    "Albus Dumbledore": "Dumbledore",
    "Albus": "Dumbledore",
    "Minerva McGonagall": "McGonagall",
    "Fleur Delacour": "Fleur",
    "Diggory":"Cedric",
    "Dumbledore":"Dumbledore",
    "Petunia Dursley":"Petunia",
    "Petunia":"Petunia",
    "Petunia Evans":"Petunia",
    "Rubeus Hagrid":"Hagrid",
    "Hagrid":"Hagrid",
    "Rubeus":"Hagrid",
    "Hedwig":"Hedwig",
    "Hedwig the owl":"Hedwig",
    "Krum":"Krum",
    "Viktor Krum":"Krum",
    "Viktor":"Krum",
    "Victor":"Krum",
    "Loopy Lupin Loony":"Lupin",
    "Remus Lupin":"Lupin",
    "Lupin":"Lupin",
    "Remus":"Lupin",
    "Loony Lupin":"Lupin",
    "Loony":"Lupin",
    "Dora Nymphadora Tonks":"Tonks",
    "Nymphadora Tonks":"Tonks",
    "Tonks":"Tonks",
    "Nymphadora":"Tonks",
    "Dora Nymphadora Tonks Lupin":"Tonks",
    "Remus J. Lupin":"Lupin",
    "M. G. McGonagall":"McGonagall",
    "Moaning Myrtle":"Moaning Myrtle",
    "Mad Moody":"Mad Moody",
    "Alastor Moody":"Mad Moody",
    "Alastor 'Mad-Eye' Moody":"Mad Moody",
    "Alastor 'Mad Eye' Moody":"Mad Moody",
    "Alastor": "Mad Moody",
    "Moony": "Lupin",
    "Padfoot": "Sirius",
    "Prongs": "James",
    "James Potter": "James",
    "James": "James",
    "Myrtle": "Moaning Myrtle",
    "Boy Who Lived": "Harry",
    "The chosen one": "Harry",
    "Harry Potter": "Harry",
    "Harry": "Harry",
    "Tom Riddle": "Voldemort",
    "Marvolo": "Voldemort",
    "Lord Voldemort": "Voldemort",
    "Voldemort": "Voldemort",
    "He-Who-Must-Not-Be-Named": "Voldemort",
    "You-Know-Who": "Voldemort",
    "Tom Marvolo Riddle": "Voldemort",
    "Riddle": "Voldemort",
    "Tom": "Voldemort",
    "Dark Lord": "Voldemort",
    "Severus Snape": "Snape",
    "Snape": "Snape",
    "Severus": "Snape",
    "Dolores Umbridge": "Umbridge",
    "Umbridge": "Umbridge",
    "Dolores": "Umbridge",
    "Dolores Jane Umbridge": "Umbridge",
    "Ronald Weasley": "Ron",
    "Ron Weasley": "Ron",
    "Ron": "Ron",
    "Ronald": "Ron",
    "Fleur Delacour-Weasley": "Fleur",
    "Fleur Delacour Weasley": "Fleur",
    "Fleur": "Fleur",
    "Hermione Granger": "Hermione",
    "Hermione": "Hermione",
    "Granger": "Hermione",
    "Hermione Jean Granger": "Hermione",
    "Hermione J. Granger": "Hermione",
    "Hermione Granger-Weasley": "Hermione",
    "Hermione Granger Weasley": "Hermione",
    "Wormtail": "Wormtail",
    "Peter Pettigrew": "Wormtail",
    "Pettigrew": "Wormtail",
    "Peter": "Wormtail",
    "Victor Krum": "Krum",
    "Mr. Dursley": "Mr. Dursley",
    "Mrs. Dursley": "Mrs. Dursley",
    "Lovegood": "Luna",
    "Gilbert Wimple": "Wimple"
}


In [14]:
df.tail()

,Unnamed: 0,full_name,categories,name
685,685.0,Zograf,['Quidditch Player'],Zograf
686,686.0,Mr and Mrs Granger,NaN,Mr. Granger
687,NaN,Mrs Granger,NaN,Mrs. Granger
688,78.0,Karl Broadmoor,['Quidditch Player'],Karl Broadmoor
689,633.0,George Weasley,"['gryff', 'Related to the Blacks', 'Order of t...",George


In [16]:
df.head()

,Unnamed: 0,full_name,categories,name
0,0.0,Hannah Abbot,"['Hufflepuff', 'Dumbledore’s Army']",Hannah
1,1.0,Euan Abercrombie,['Gryffindor'],Euan
2,2.0,Steward Ackerley,['Ravenclaw'],Steward
3,3.0,Falcon Aesalon,['Chocolate Frog Card'],Falcon
4,4.0,Agnes,[],Agnes


In [18]:
import pandas as pd

# df = your dataframe
# name_corrections = your dictionary

# Make sure full_name is string
df["full_name"] = df["full_name"].astype(str)

# Track rows to add later
rows_to_add = []

for key, corrected_name in name_corrections.items():

    # Find rows where full_name exactly matches the key
    mask = df["name"] == key

    if not mask.any():
        # Row does not exist -> create it
        rows_to_add.append({
            "Unnamed: 0": None,
            "name": key,
            "categories": [],
            "full_name": corrected_name
        })

# Add missing rows
if rows_to_add:
    df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)

# Reset index for cleanliness
df.reset_index(drop=True, inplace=True)


/var/folders/dp/1z2xwn7516q1b4n5t3nlr0500000gn/T/ipykernel_1871/3825662657.py:28: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame(rows_to_add)], ignore_index=True)


In [20]:
df.to_csv("../data/processed/characters_final_3.csv", index=False)